# Deploy the graph agent with the AWS CLI

Deploys Demo 02 one resource at a time with the AWS CLI, from `%%bash` cells: the same agent as Demo 01 on
[Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) (platform version V2, code package, no container), Tavily behind
[AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-integrations.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) and the knowledge graph in Neo4j AuraDB. It creates exactly what
`deploy_with_boto3.ipynb` creates (with the prefix `tavily-graph-cli`), so you can read the two side by side.

```
 you ──invoke-agent-runtime──▶ Runtime (V2, research_agent.py) ──SigV4──▶ Gateway ──API key──▶ api.tavily.com
                                    │  └─ Bedrock model                      └─ key kept in AgentCore Identity
                                    └─ Secrets Manager ──▶ Neo4j AuraDB credentials
```

| # | Resource | Command | CLI reference |
|---|----------|---------|---------------|
| 2 | Secrets Manager secret (AuraDB credentials) | `aws secretsmanager create-secret` | [create-secret](https://docs.aws.amazon.com/cli/latest/reference/secretsmanager/create-secret.html) |
| 3 | AgentCore Identity API key credential provider (Tavily) | `aws bedrock-agentcore-control create-api-key-credential-provider` | [create-api-key-credential-provider](https://docs.aws.amazon.com/cli/latest/reference/bedrock-agentcore-control/create-api-key-credential-provider.html) |
| 4 | IAM role for the Gateway | `aws iam create-role`, `put-role-policy` | [create-role](https://docs.aws.amazon.com/cli/latest/reference/iam/create-role.html) |
| 5 | Gateway (MCP, IAM inbound) | `aws bedrock-agentcore-control create-gateway` | [create-gateway](https://docs.aws.amazon.com/cli/latest/reference/bedrock-agentcore-control/create-gateway.html) |
| 6 | Gateway target `tavily` (OpenAPI) | `aws bedrock-agentcore-control create-gateway-target` | [create-gateway-target](https://docs.aws.amazon.com/cli/latest/reference/bedrock-agentcore-control/create-gateway-target.html) |
| 7 | S3 bucket + code package | `aws s3api create-bucket`, `put-object` | [create-bucket](https://docs.aws.amazon.com/cli/latest/reference/s3api/create-bucket.html) |
| 8 | IAM execution role for the Runtime | `aws iam create-role`, `put-role-policy` | [Runtime permissions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) |
| 9 | AgentCore Runtime (V2) | `aws bedrock-agentcore-control create-agent-runtime` | [create-agent-runtime](https://docs.aws.amazon.com/cli/latest/reference/bedrock-agentcore-control/create-agent-runtime.html) |
| 10 | Ask the agent | `aws bedrock-agentcore invoke-agent-runtime` | [invoke-agent-runtime](https://docs.aws.amazon.com/cli/latest/reference/bedrock-agentcore/invoke-agent-runtime.html) |
| 11 | Cleanup, in reverse order | `delete-*` | |

**Before you run it**
- AWS CLI v2 **2.36.46 or later**: that release added `--platform-version` to `create-agent-runtime`
  ([CLI changelog](https://github.com/aws/aws-cli/blob/v2/CHANGELOG.rst)). Check with `aws --version`.
- `uv`, `zip` and `python3` (the build script and a few one-line JSON helpers use them), AWS credentials that can create the resources above, and a Bedrock model you can invoke.
- Exported in the shell that started Jupyter: `TAVILY_API_KEY` (or `TAVILY_KEY`) and your AuraDB `NEO4J_URI`, `NEO4J_USERNAME`
  (or `NEO4J_USER`), `NEO4J_PASSWORD`, `NEO4J_DATABASE`. Secret values go through files with mode 600 that are deleted right after the call, never through command-line arguments.
- Each `%%bash` cell is its own process, so values that later cells need (IDs, ARNs) are saved in `build/cli/state.env`, a git-ignored file. Run the cells top to bottom.
  If one fails halfway, run section 11 (it finds everything by name) and start again.
- Cost: AgentCore Runtime and Gateway are pay-per-use; Secrets Manager charges per secret per month. Run section 11 when you finish.

## 1. Configuration and helpers

The first cell writes names, Region and the account (read with `aws sts get-caller-identity`, so no account ID is written in the notebook) to
`build/cli/state.env`. The second cell writes `helpers.sh`, sourced by every later cell:
- `mask` hides the account ID in printed output.
- `save KEY value` appends a value to the state file.
- `wait_ready` and `wait_gone` poll a command until the status is `READY` or the resource is gone. The CLI has `aws ... wait` only for some
  AgentCore resources (`aws bedrock-agentcore-control wait help` lists them: memory, policy), not for gateways, targets or runtimes, so these
  are loops over `get-*` with `--query status`. The docs poll the runtime the same way ([V2 create and update take minutes](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)).

Runtime names allow only letters, digits and underscores; Gateway names allow hyphens.

In [ ]:
%%bash
set -euo pipefail
: "${TAVILY_API_KEY:=${TAVILY_KEY:-}}"
: "${NEO4J_URI:=}" "${NEO4J_PASSWORD:=}"
for name in TAVILY_API_KEY NEO4J_URI NEO4J_PASSWORD; do
  [ -n "${!name}" ] || { echo "export $name in the shell that started Jupyter (TAVILY_KEY also works for the Tavily key)" >&2; exit 1; }
done

mkdir -p build/cli
AWS_REGION="${AWS_REGION:-us-east-1}"
ACCOUNT=$(aws sts get-caller-identity --query Account --output text)

cat > build/cli/state.env <<EOF
AWS_REGION=$AWS_REGION
ACCOUNT=$ACCOUNT
NEO4J_SECRET_NAME=tavily-graph-cli/neo4j-aura
KEY_PROVIDER_NAME=tavily-graph-cli-tavily-key
GATEWAY_ROLE_NAME=tavily-graph-cli-gateway-role
GATEWAY_NAME=tavily-graph-cli
TARGET_NAME=tavily
RUNTIME_ROLE_NAME=tavily-graph-cli-runtime-role
RUNTIME_NAME=tavily_graph_cli
ENTRYPOINT=research_agent.py
CODE_KEY=research-agent/agent-code.zip
BUCKET=tavily-graph-cli-code-$ACCOUNT-$AWS_REGION
BEDROCK_MODEL_ID=${BEDROCK_MODEL_ID:-us.anthropic.claude-sonnet-4-5-20250929-v1:0}
EOF
echo "Region: $AWS_REGION | account: <account> | state file: build/cli/state.env"
aws --version

In [ ]:
%%bash
cat > build/cli/helpers.sh <<'EOF'
# Sourced by every cell: names and IDs from the state file, plus small helpers.
set -euo pipefail
export AWS_PAGER=""
STATE=build/cli/state.env
set -a; source "$STATE"; set +a
export AWS_DEFAULT_REGION="$AWS_REGION"

mask() { sed "s/$ACCOUNT/<account>/g"; }              # hide the account ID in printed output
save() { echo "$1=$2" >> "$STATE"; }                   # remember a value for later cells (the last line wins)

wait_ready() {  # wait_ready "<what>" <command that prints the status>
  local what="$1"; shift; local status
  for _ in $(seq 1 180); do
    status=$("$@")
    echo "  $what: $status"
    case "$status" in READY) return 0 ;; *FAILED|*UNSUCCESSFUL) return 1 ;; esac
    sleep 10
  done
  echo "$what: timed out" >&2; return 1
}

wait_gone() {   # wait_gone "<what>" <get command that fails once the resource no longer exists>
  local what="$1"; shift
  for _ in $(seq 1 90); do
    if "$@" >/dev/null 2>&1; then echo "  $what: deleting"; sleep 10; else echo "  $what: deleted"; return 0; fi
  done
  echo "$what: still exists" >&2; return 1
}
EOF
echo "helpers written"

## 2. Secrets Manager secret for the AuraDB credentials

**What:** a secret with `{"uri", "username", "password", "database"}`. **Why:** the agent reads it at startup (`NEO4J_SECRET_ID`), so the
password is never in the code package, the runtime environment variables or a template. The runtime role (section 8) gets
`secretsmanager:GetSecretValue` on this one secret. The value is written to a mode-600 file, passed with `--secret-string file://...` and deleted.
CLI reference: [create-secret](https://docs.aws.amazon.com/cli/latest/reference/secretsmanager/create-secret.html).

In [ ]:
%%bash
source build/cli/helpers.sh
umask 077
export NEO4J_USERNAME="${NEO4J_USERNAME:-${NEO4J_USER:-neo4j}}" NEO4J_DATABASE="${NEO4J_DATABASE:-neo4j}"
python3 -c 'import json, os; print(json.dumps({"uri": os.environ["NEO4J_URI"], "username": os.environ["NEO4J_USERNAME"], "password": os.environ["NEO4J_PASSWORD"], "database": os.environ["NEO4J_DATABASE"]}))' > build/cli/neo4j-secret.json

NEO4J_SECRET_ARN=$(aws secretsmanager create-secret \
  --name "$NEO4J_SECRET_NAME" \
  --description "Neo4j AuraDB connection for the graph agent demo" \
  --secret-string file://build/cli/neo4j-secret.json \
  --query ARN --output text)
rm -f build/cli/neo4j-secret.json
save NEO4J_SECRET_ARN "$NEO4J_SECRET_ARN"
echo "Secret created: $(echo "$NEO4J_SECRET_ARN" | mask)"

## 3. AgentCore Identity API key credential provider for Tavily

**What:** a credential provider in AgentCore Identity that stores the Tavily API key. **Why:** the Gateway target (section 6) references the provider
by ARN and the Gateway adds the key to each request to `api.tavily.com`; the agent never sees the key. AgentCore keeps the key in a secret it
manages and returns that secret's ARN, which the Gateway role needs in section 4. The key goes in through `--cli-input-json file://...`, not as an argument
(arguments show up in `ps`).
Docs: [Set up outbound authorization with an API key](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-outbound-auth.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

In [ ]:
%%bash
source build/cli/helpers.sh
umask 077
export TAVILY_API_KEY="${TAVILY_API_KEY:-${TAVILY_KEY:?export TAVILY_API_KEY}}"
python3 -c 'import json, os; print(json.dumps({"name": os.environ["KEY_PROVIDER_NAME"], "apiKey": os.environ["TAVILY_API_KEY"]}))' > build/cli/provider.json

read -r PROVIDER_ARN PROVIDER_SECRET_ARN < <(aws bedrock-agentcore-control create-api-key-credential-provider \
  --cli-input-json file://build/cli/provider.json \
  --query '[credentialProviderArn, apiKeySecretArn.secretArn]' --output text)
rm -f build/cli/provider.json
save PROVIDER_ARN "$PROVIDER_ARN"; save PROVIDER_SECRET_ARN "$PROVIDER_SECRET_ARN"
echo "Credential provider created: $(echo "$PROVIDER_ARN" | mask)"

## 4. IAM role for the Gateway

**What:** the service role the Gateway assumes. The trust policy lets only `bedrock-agentcore.amazonaws.com` assume it, and only for gateways in this
account whose ID starts with the gateway name (a gateway ID is `<name>-<random suffix>`, so the exact ARN does not exist yet). **Why:** to add the API key to
the outbound call the Gateway must read it. The docs list three permissions for API key outbound auth: `GetWorkloadAccessToken` (on the workload identity of
this gateway), `GetResourceApiKey` (on the credential provider) and `secretsmanager:GetSecretValue` (on the secret AgentCore created for the key).
Docs: [Gateway service role](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-prerequisites-permissions.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el), [API key outbound auth](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-outbound-auth.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).
IAM is eventually consistent, so the cell waits a few seconds
([IAM troubleshooting](https://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_general.html#troubleshoot_general_eventual-consistency)).

In [ ]:
%%bash
source build/cli/helpers.sh

cat > build/cli/gateway-trust.json <<EOF
{"Version": "2012-10-17", "Statement": [{
  "Sid": "AssumeRolePolicy", "Effect": "Allow",
  "Principal": {"Service": "bedrock-agentcore.amazonaws.com"}, "Action": "sts:AssumeRole",
  "Condition": {"StringEquals": {"aws:SourceAccount": "$ACCOUNT"},
                "ArnLike": {"aws:SourceArn": "arn:aws:bedrock-agentcore:$AWS_REGION:$ACCOUNT:gateway/$GATEWAY_NAME-*"}}}]}
EOF

cat > build/cli/gateway-policy.json <<EOF
{"Version": "2012-10-17", "Statement": [
  {"Sid": "GetWorkloadAccessToken", "Effect": "Allow", "Action": "bedrock-agentcore:GetWorkloadAccessToken",
   "Resource": ["arn:aws:bedrock-agentcore:$AWS_REGION:$ACCOUNT:workload-identity-directory/default",
                "arn:aws:bedrock-agentcore:$AWS_REGION:$ACCOUNT:workload-identity-directory/default/workload-identity/$GATEWAY_NAME-*"]},
  {"Sid": "GetResourceApiKey", "Effect": "Allow", "Action": "bedrock-agentcore:GetResourceApiKey",
   "Resource": ["arn:aws:bedrock-agentcore:$AWS_REGION:$ACCOUNT:token-vault/default", "$PROVIDER_ARN"]},
  {"Sid": "GetSecretValue", "Effect": "Allow", "Action": "secretsmanager:GetSecretValue", "Resource": "$PROVIDER_SECRET_ARN"}]}
EOF

GATEWAY_ROLE_ARN=$(aws iam create-role \
  --role-name "$GATEWAY_ROLE_NAME" \
  --description "AgentCore Gateway service role for the Tavily target (graph agent demo)" \
  --assume-role-policy-document file://build/cli/gateway-trust.json \
  --query Role.Arn --output text)
aws iam put-role-policy \
  --role-name "$GATEWAY_ROLE_NAME" \
  --policy-name tavily-api-key-outbound-auth \
  --policy-document file://build/cli/gateway-policy.json
save GATEWAY_ROLE_ARN "$GATEWAY_ROLE_ARN"
echo "Gateway role created: $(echo "$GATEWAY_ROLE_ARN" | mask)"
sleep 10

## 5. Gateway

**What:** an MCP gateway with `AWS_IAM` inbound authorization. **Why:** the agent calls it with its own IAM credentials (SigV4), so there is no OAuth
server to run; callers need `bedrock-agentcore:InvokeGateway` on this gateway (granted in section 8). `create-gateway` returns while the gateway is
`CREATING`, so the cell polls `get-gateway` until `READY`.
Docs: [Create a gateway with the API](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-create-api.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el), [inbound authorization](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-inbound-auth.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

In [ ]:
%%bash
source build/cli/helpers.sh

GATEWAY_ID=$(aws bedrock-agentcore-control create-gateway \
  --name "$GATEWAY_NAME" \
  --description "Tavily web search and extract as MCP tools" \
  --role-arn "$GATEWAY_ROLE_ARN" \
  --protocol-type MCP \
  --authorizer-type AWS_IAM \
  --query gatewayId --output text)
save GATEWAY_ID "$GATEWAY_ID"

wait_ready gateway aws bedrock-agentcore-control get-gateway --gateway-identifier "$GATEWAY_ID" --query status --output text
read -r GATEWAY_ARN GATEWAY_URL < <(aws bedrock-agentcore-control get-gateway \
  --gateway-identifier "$GATEWAY_ID" --query '[gatewayArn, gatewayUrl]' --output text)
save GATEWAY_ARN "$GATEWAY_ARN"; save GATEWAY_URL "$GATEWAY_URL"
echo "Gateway ready: $(echo "$GATEWAY_URL" | mask)"

## 6. Gateway target `tavily`

**What:** an OpenAPI target built from `schemas/tavily-openapi.json`, with the API key from section 3 sent as the header `Authorization: Bearer <key>`.
**Why:** AgentCore documents a *Tavily Search* integration template (server `https://api.tavily.com`, API key auth, `/search` and `/extract`), but only in the console ([integration templates](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-integrations.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)). The API has no template
field, so this target describes the same server as an OpenAPI schema. We write the operations ourselves (`search_web`, `read_web_page`):
the model reads each `operationId` and `description` as the tool's name and description on every call, so they say when to use the tool
and when not, and match the local notebook's `@tool` functions. The schema follows the [OpenAPI target rules](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-schema-openapi.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el):
`operationId` on every operation, JSON only, no `oneOf`/`anyOf`/`allOf`, no security schemes (auth lives in the credential configuration).
The Gateway names each tool `<target>___<operationId>` ([tool naming](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-tool-naming.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)), which is what the agent's tool filter `^tavily___` expects.
`inlinePayload` takes the schema as a string, so the small Python helper embeds the file's text in the JSON.

In [ ]:
%%bash
source build/cli/helpers.sh

python3 -c 'import json; print(json.dumps({"mcp": {"openApiSchema": {"inlinePayload": open("schemas/tavily-openapi.json").read()}}}))' > build/cli/target-config.json
cat > build/cli/target-credentials.json <<EOF
[{"credentialProviderType": "API_KEY",
  "credentialProvider": {"apiKeyCredentialProvider": {
    "providerArn": "$PROVIDER_ARN", "credentialLocation": "HEADER",
    "credentialParameterName": "Authorization", "credentialPrefix": "Bearer "}}}]
EOF

TARGET_ID=$(aws bedrock-agentcore-control create-gateway-target \
  --gateway-identifier "$GATEWAY_ID" \
  --name "$TARGET_NAME" \
  --description "Web tools: search_web (/search) and read_web_page (/extract)" \
  --target-configuration file://build/cli/target-config.json \
  --credential-provider-configurations file://build/cli/target-credentials.json \
  --query targetId --output text)
save TARGET_ID "$TARGET_ID"
wait_ready target aws bedrock-agentcore-control get-gateway-target --gateway-identifier "$GATEWAY_ID" --target-id "$TARGET_ID" --query status --output text

## 7. Code package and S3 bucket

**What:** `build_package.sh` installs the agent's dependencies as **linux/arm64** wheels (`uv pip install --python-platform aarch64-manylinux2014
--python-version 3.12 --only-binary=:all:`), copies `agent/*.py` next to them and zips the result (limits: 250 MB zipped, 750 MB unzipped). No Docker.
The zip goes to a private S3 bucket (all public access blocked), because `create-agent-runtime` reads the code from S3. `--expected-bucket-owner`
makes the upload fail if the bucket belongs to someone else.
Docs: [Direct code deployment for Python](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-get-started-code-deploy-python.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

In [ ]:
%%bash
source build/cli/helpers.sh

bash build_package.sh

# us-east-1 takes no LocationConstraint; every other Region requires one.
if [ "$AWS_REGION" = us-east-1 ]; then
  aws s3api create-bucket --bucket "$BUCKET" > /dev/null
else
  aws s3api create-bucket --bucket "$BUCKET" --create-bucket-configuration LocationConstraint="$AWS_REGION" > /dev/null
fi
aws s3api put-public-access-block \
  --bucket "$BUCKET" \
  --public-access-block-configuration BlockPublicAcls=true,IgnorePublicAcls=true,BlockPublicPolicy=true,RestrictPublicBuckets=true
aws s3api put-object \
  --bucket "$BUCKET" --key "$CODE_KEY" --body build/agent-code.zip \
  --expected-bucket-owner "$ACCOUNT" > /dev/null
echo "Uploaded build/agent-code.zip to s3://<bucket>/$CODE_KEY"

## 8. IAM execution role for the Runtime

**What:** the role the agent runs as. It starts from the docs' *direct deploy execution role* (CloudWatch Logs, X-Ray, CloudWatch metrics, Bedrock model
invocation) and adds only what this agent calls: `s3:GetObject` on the one code object, `bedrock-agentcore:InvokeGateway` on the one gateway, and
`secretsmanager:GetSecretValue` on the one AuraDB secret. The model statement allows foundation models and this account's inference profiles, because Claude
Sonnet 4.5 is invoked through a `us.` inference profile. The agent does not use AgentCore Identity tokens itself (the Gateway does), so the workload-identity
statement from the full role is left out.
Docs: [Runtime permissions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

In [ ]:
%%bash
source build/cli/helpers.sh
LOGS="arn:aws:logs:$AWS_REGION:$ACCOUNT:log-group:/aws/bedrock-agentcore/runtimes"

cat > build/cli/runtime-trust.json <<EOF
{"Version": "2012-10-17", "Statement": [{
  "Sid": "AssumeRolePolicy", "Effect": "Allow",
  "Principal": {"Service": "bedrock-agentcore.amazonaws.com"}, "Action": "sts:AssumeRole",
  "Condition": {"StringEquals": {"aws:SourceAccount": "$ACCOUNT"},
                "ArnLike": {"aws:SourceArn": "arn:aws:bedrock-agentcore:$AWS_REGION:$ACCOUNT:*"}}}]}
EOF

cat > build/cli/runtime-policy.json <<EOF
{"Version": "2012-10-17", "Statement": [
  {"Effect": "Allow", "Action": ["logs:DescribeLogStreams", "logs:CreateLogGroup"], "Resource": "$LOGS/*"},
  {"Effect": "Allow", "Action": "logs:PutResourcePolicy", "Resource": "$LOGS/$RUNTIME_NAME-*"},
  {"Effect": "Allow", "Action": "logs:DescribeLogGroups", "Resource": "arn:aws:logs:$AWS_REGION:$ACCOUNT:log-group:*"},
  {"Effect": "Allow", "Action": ["logs:CreateLogStream", "logs:PutLogEvents"], "Resource": "$LOGS/*:log-stream:*"},
  {"Effect": "Allow", "Action": ["xray:PutTraceSegments", "xray:PutTelemetryRecords", "xray:GetSamplingRules", "xray:GetSamplingTargets"], "Resource": "*"},
  {"Effect": "Allow", "Action": "cloudwatch:PutMetricData", "Resource": "*",
   "Condition": {"StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}}},
  {"Sid": "BedrockModelInvocation", "Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
   "Resource": ["arn:aws:bedrock:*::foundation-model/*", "arn:aws:bedrock:$AWS_REGION:$ACCOUNT:inference-profile/*"]},
  {"Sid": "ReadCodePackage", "Effect": "Allow", "Action": "s3:GetObject", "Resource": "arn:aws:s3:::$BUCKET/$CODE_KEY"},
  {"Sid": "InvokeTavilyGateway", "Effect": "Allow", "Action": "bedrock-agentcore:InvokeGateway", "Resource": "$GATEWAY_ARN"},
  {"Sid": "ReadNeo4jSecret", "Effect": "Allow", "Action": "secretsmanager:GetSecretValue", "Resource": "$NEO4J_SECRET_ARN"}]}
EOF

RUNTIME_ROLE_ARN=$(aws iam create-role \
  --role-name "$RUNTIME_ROLE_NAME" \
  --description "AgentCore Runtime execution role for the graph agent demo" \
  --assume-role-policy-document file://build/cli/runtime-trust.json \
  --query Role.Arn --output text)
aws iam put-role-policy \
  --role-name "$RUNTIME_ROLE_NAME" \
  --policy-name graph-agent-runtime \
  --policy-document file://build/cli/runtime-policy.json
save RUNTIME_ROLE_ARN "$RUNTIME_ROLE_ARN"
echo "Runtime role created: $(echo "$RUNTIME_ROLE_ARN" | mask)"
sleep 10

## 9. AgentCore Runtime (platform version V2)

**What:** the runtime, from the code package. `entryPoint` is an array of files to run, so the file can be named `research_agent.py`; `main.py` is only the
example name in the docs. `--platform-version V2` starts the agent from a snapshot taken after startup (V1 is the default if you omit it). **Why V2 changes
the code:** work done at startup is frozen in the snapshot, so `research_agent.py` builds the model, the Neo4j driver and the graph store at module level, and
builds the Gateway client and the `Agent` inside the request handler ([Optimize your agent for V2](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-v2-optimize.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)). The agent must report
healthy within 120 s of starting.

`create-agent-runtime` returns while the runtime is `CREATING`. A V2 create **takes several minutes**; calling update or delete early returns `ConflictException`,
so the cell polls `get-agent-runtime`. Network mode `PUBLIC` lets the agent reach the Gateway, Secrets Manager and AuraDB. The lifecycle settings keep their
defaults (15 minutes idle, 8 hours maximum session).
Docs: [Platform versions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el), [Direct code deployment](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-get-started-code-deploy-python.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

In [ ]:
%%bash
source build/cli/helpers.sh

cat > build/cli/runtime-artifact.json <<EOF
{"codeConfiguration": {"code": {"s3": {"bucket": "$BUCKET", "prefix": "$CODE_KEY"}},
                       "runtime": "PYTHON_3_12", "entryPoint": ["$ENTRYPOINT"]}}
EOF
cat > build/cli/runtime-env.json <<EOF
{"TAVILY_GATEWAY_URL": "$GATEWAY_URL", "NEO4J_SECRET_ID": "$NEO4J_SECRET_NAME", "BEDROCK_MODEL_ID": "$BEDROCK_MODEL_ID"}
EOF

RUNTIME_ID=$(aws bedrock-agentcore-control create-agent-runtime \
  --agent-runtime-name "$RUNTIME_NAME" \
  --description "Research agent with a Neo4j knowledge graph; Tavily through AgentCore Gateway" \
  --agent-runtime-artifact file://build/cli/runtime-artifact.json \
  --role-arn "$RUNTIME_ROLE_ARN" \
  --network-configuration '{"networkMode": "PUBLIC"}' \
  --platform-version V2 \
  --environment-variables file://build/cli/runtime-env.json \
  --query agentRuntimeId --output text)
save RUNTIME_ID "$RUNTIME_ID"

wait_ready runtime aws bedrock-agentcore-control get-agent-runtime --agent-runtime-id "$RUNTIME_ID" --query status --output text
RUNTIME_ARN=$(aws bedrock-agentcore-control get-agent-runtime --agent-runtime-id "$RUNTIME_ID" --query agentRuntimeArn --output text)
save RUNTIME_ARN "$RUNTIME_ARN"
echo "Platform version: $(aws bedrock-agentcore-control get-agent-runtime --agent-runtime-id "$RUNTIME_ID" --query platformVersion --output text)"
echo "Runtime: $(echo "$RUNTIME_ARN" | mask)"

## 10. Ask the agent

**What:** `invoke-agent-runtime` with `{"prompt": ...}`; the answer is written to a file. **Why a new `--runtime-session-id` per question:** each session is its own
microVM, so nothing the agent "remembers" survives between questions except the Neo4j graph. That is the point of the demo: the first questions fill the graph,
the later ones are answered from it. The session ID must be at least 33 characters, so a UUID string works. `--payload` is a blob, so it takes `fileb://`.
The first call after creation can take longer (it restores the snapshot and opens the Gateway connection). The five questions come from `invoke_agent.py`.
CLI reference: [invoke-agent-runtime](https://docs.aws.amazon.com/cli/latest/reference/bedrock-agentcore/invoke-agent-runtime.html).

Expect `tools_used` to contain `tavily___search_web` for the first questions and fewer or no searches for the last ones. Search counts are model behavior
and change from run to run.

In [ ]:
%%bash
source build/cli/helpers.sh

python3 -c 'from invoke_agent import SERIES; print("\n".join(SERIES))' | while IFS= read -r QUESTION; do
  python3 -c 'import json, sys; print(json.dumps({"prompt": sys.argv[1]}))' "$QUESTION" > build/cli/payload.json
  aws bedrock-agentcore invoke-agent-runtime \
    --agent-runtime-arn "$RUNTIME_ARN" \
    --runtime-session-id "$(python3 -c 'import uuid; print(uuid.uuid4())')" \
    --content-type application/json \
    --accept application/json \
    --payload fileb://build/cli/payload.json \
    build/cli/answer.json < /dev/null > /dev/null
  echo; echo "Q: $QUESTION"
  python3 -c 'import json; r = json.load(open("build/cli/answer.json")); print("  tools used:", r["tools_used"]); print(" ", r["answer"])'
done

## 11. Cleanup

Deletes everything this notebook created, **in reverse order**: runtime, gateway target, gateway, credential provider, runtime role, gateway role, code object and
bucket, secret. It finds each resource by name, so it works after a restart or a half-finished run. Your AuraDB instance and the graph in it are not touched.

Nothing is deleted until you change `CONFIRM_DELETE=false` to `CONFIRM_DELETE=true` at the top of the cell and run it. With `false` it only prints each command.
A runtime deletes asynchronously, and V2 can keep its snapshot for up to 8 hours after deletion ([snapshot lifecycle](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)); AgentCore manages that.

In [ ]:
%%bash
CONFIRM_DELETE=false        # change to true to delete the resources of this notebook

source build/cli/helpers.sh
run() { if [ "$CONFIRM_DELETE" = true ]; then "$@"; else echo "[dry run] $*" | mask >&2; fi; }
gone() { [ "$CONFIRM_DELETE" = true ] && wait_gone "$@" || true; }

# 1. Runtime
RUNTIME_ID=$(aws bedrock-agentcore-control list-agent-runtimes \
  --query "agentRuntimes[?agentRuntimeName=='$RUNTIME_NAME'].agentRuntimeId | [0]" --output text)
if [ "$RUNTIME_ID" != None ]; then
  run aws bedrock-agentcore-control delete-agent-runtime --agent-runtime-id "$RUNTIME_ID" > /dev/null
  gone runtime aws bedrock-agentcore-control get-agent-runtime --agent-runtime-id "$RUNTIME_ID"
fi

# 2. Gateway targets, then the gateway
GATEWAY_ID=$(aws bedrock-agentcore-control list-gateways \
  --query "items[?name=='$GATEWAY_NAME'].gatewayId | [0]" --output text)
if [ "$GATEWAY_ID" != None ]; then
  for TARGET in $(aws bedrock-agentcore-control list-gateway-targets --gateway-identifier "$GATEWAY_ID" --query 'items[].targetId' --output text); do
    run aws bedrock-agentcore-control delete-gateway-target --gateway-identifier "$GATEWAY_ID" --target-id "$TARGET" > /dev/null
    gone target aws bedrock-agentcore-control get-gateway-target --gateway-identifier "$GATEWAY_ID" --target-id "$TARGET"
  done
  run aws bedrock-agentcore-control delete-gateway --gateway-identifier "$GATEWAY_ID" > /dev/null
  gone gateway aws bedrock-agentcore-control get-gateway --gateway-identifier "$GATEWAY_ID"
fi

# 3. Credential provider (AgentCore Identity)
run aws bedrock-agentcore-control delete-api-key-credential-provider --name "$KEY_PROVIDER_NAME" || echo "credential provider already gone"

# 4. IAM roles (inline policies first)
for ROLE in "$RUNTIME_ROLE_NAME" "$GATEWAY_ROLE_NAME"; do
  for POLICY in $(aws iam list-role-policies --role-name "$ROLE" --query 'PolicyNames[]' --output text 2>/dev/null || true); do
    run aws iam delete-role-policy --role-name "$ROLE" --policy-name "$POLICY"
  done
  run aws iam delete-role --role-name "$ROLE" || echo "role $ROLE already gone"
done

# 5. Code package and bucket
run aws s3api delete-object --bucket "$BUCKET" --key "$CODE_KEY" --expected-bucket-owner "$ACCOUNT" > /dev/null || echo "code object already gone"
run aws s3api delete-bucket --bucket "$BUCKET" --expected-bucket-owner "$ACCOUNT" || echo "bucket already gone"

# 6. AuraDB secret (no recovery window, so the name can be reused at once)
run aws secretsmanager delete-secret --secret-id "$NEO4J_SECRET_NAME" --force-delete-without-recovery > /dev/null || echo "secret already gone"

if [ "$CONFIRM_DELETE" = true ]; then rm -rf build/cli; echo "Done. Removed build/cli."; else echo "Dry run only: set CONFIRM_DELETE=true to delete."; fi